# getitrack: Real-Video Multi-Object Tracking

Track objects in real videos with **RF-DETR** detection and getitrack's **ByteTrack**.
Runs end to end on Google Colab (or locally).

Pipeline: **frame -> detector -> `Detections` -> ByteTrack -> annotated video**. The
detector sits behind a getitrack `DetectionAdapter`, so any detector plugs in the same way.

## 1. Install

In [1]:
%%writefile overrides.txt
opencv-python-headless~=4.11
pyarrow>=24.0,<25
idna>=3.15

Writing overrides.txt


In [2]:
!pip install -q uv
!uv pip install -q --system --override overrides.txt \
  "getitune @ git+https://github.com/omkar-334/otx.git@getitrack-demo#subdirectory=library" \
  "getitrack @ git+https://github.com/omkar-334/otx.git@getitrack-demo#subdirectory=libraries/getitrack"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 82.3 MB/s eta 0:00:00


In [ ]:
import os

os.kill(os.getpid(), 9)

## 2. Download demo videos

Three short clips (bicycles/motorcycles, airplanes, apples) from the project release.

In [1]:
import urllib.request
from pathlib import Path

Path("videos").mkdir(exist_ok=True)
BASE = "https://github.com/omkar-334/otx/releases/download/demo-videos"
for name in ["bikes-1", "jets-1", "apples"]:
    dst = Path("videos") / f"{name}.mp4"
    if not dst.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}.mp4", dst)
    print("ready", dst)

ready videos/bikes-1.mp4
ready videos/jets-1.mp4
ready videos/apples.mp4


## 3. Detector adapter

`GetiAdapter` wraps getitune backend behind getitrack's `DetectionAdapter` interface:
`detect(frame, frame_id) -> Detections`. It just detects; which classes to *track* is
chosen later via `ByteTrackConfig.class_filter`.

In [2]:
import torch
from getitune.backend.lightning.models.detection.yolox import YOLOX

from getitrack.adapters import GetiAdapter
from getitrack.utils import COCO_CLASSES

device = "cuda" if torch.cuda.is_available() else "cpu"
yolox = YOLOX(label_info=80, model_name="yolox_tiny").eval().to(device)
yolox.hparams["best_confidence_threshold"] = 0.1
detector = GetiAdapter(yolox, device=device)

# YOLOX predicts contiguous 80-class COCO ids (0-79); map them to names.
COCO80 = dict(enumerate(COCO_CLASSES.values()))

Downloading: "https://storage.geti.intel.com/weights/yolox_tiny_8x8.pth" to /root/.cache/torch/hub/checkpoints/yolox_tiny_8x8.pth


## 4. Tracking pipeline

For each frame: detect, `tracker.update`, draw. `class_filter` keeps only the classes
we care about. Note the loop never mentions the detector, only `adapter.detect(...)`.

In [16]:
from pathlib import Path
from tqdm import tqdm

import cv2

from getitrack import BaseTracker, ByteTrackConfig, TrackAnnotator, to_h264
from getitrack.io import VideoReader, VideoWriter


def track_video(detector, video, output, width=960, class_names=COCO80):
    video, output = Path(video), Path(output)
    tracker = BaseTracker.from_config(ByteTrackConfig())
    annotator = TrackAnnotator(show_score=True, class_names=class_names)
    with VideoReader(video) as reader:
        size = (width, round(reader.height * width / reader.width))
        raw = output.with_name(f"{output.stem}.raw.mp4")
        with VideoWriter(raw, fps=reader.fps or 30.0, frame_size=size) as writer:
            for frame_id, frame in enumerate(
                tqdm(reader, desc=video.name, unit="frame")
            ):
                resized = cv2.resize(frame, size)
                tracked = tracker.update(detector.detect(resized, frame_id))
                writer.write(annotator.annotate(resized, tracked))
    final = to_h264(raw, output)
    raw.unlink(missing_ok=True)
    print(f"{video.name}: {writer.frames_written} frames -> {final}")
    return final

## 5. Run and watch

Build the detector once, then track each clip for the class it contains.

In [5]:
from IPython.display import Video, display

# videos = ["bikes-1", "jets-1", "apples"]
name = "bikes-1"

Path("results").mkdir(exist_ok=True)

out = track_video(detector, f"videos/{name}.mp4", f"results/{name}_tracked.mp4")
display(Video(str(out), embed=True, width=720))

bikes-1.mp4: 388frame [00:19, 19.85frame/s]


bikes-1.mp4: 388 frames -> results/bikes-1_tracked.mp4


## 6. Modularity: swap the detector, keep the pipeline

getitrack is detector-agnostic. The tracking code above never names RF-DETR, it just
calls `adapter.detect(...)`. We can change the detector backend. A **getitune** model plugs into the *same* `track_video`
through the built-in `GetiAdapter`, changing only how the adapter is built.


Now we will use the `rfdetr` package as a backend - Define `RFDETRAdapter` by wrapping `DetectionAdapter` and use it for detection

In [7]:
!uv pip install --system rfdetr

Using Python 3.12.13 environment at: /usr
Resolved 85 packages in 256ms
Prepared 2 packages in 1.38s
Uninstalled 2 packages in 18ms
Installed 2 packages in 6ms
 - idna==3.18
 + idna==3.7
 - opencv-python-headless==4.13.0.92
 + opencv-python-headless==4.10.0.84


In [21]:
from __future__ import annotations

from typing import TYPE_CHECKING

import cv2
import numpy as np

from getitrack.adapters.base import DetectionAdapter
from getitrack.core.detection import Detections

from rfdetr.detr import RFDETR
from rfdetr.util.coco_classes import COCO_CLASSES

class RFDETRAdapter(DetectionAdapter):
    """Runs a Roboflow RF-DETR model on raw BGR frames."""
    def __init__(self, model: RFDETR, score_thresh: float = 0.4) -> None:
        """Wrap an RF-DETR model.

        Args:
            model: An RF-DETR model instance, e.g. ``
            score_thresh: Minimum detection confidence to keep.
        """
        self.model = model
        self.score_thresh = score_thresh

    @property
    def class_names(self) -> dict[int, str]:
        """RF-DETR's own COCO class-id-to-name lookup"""
        from rfdetr.util.coco_classes import COCO_CLASSES

        return COCO_CLASSES

    def detect(self, frame_bgr: np.ndarray, frame_id: int) -> Detections:
        """Run one BGR frame through RF-DETR.

        Args:
            frame_bgr: ``(H, W, 3)`` uint8 frame in BGR order.
            frame_id: Frame index to stamp on the ret

        Returns:
            `Detections` in original frame coordinates.
        """
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        det = self.model.predict(rgb, threshold=self.score_thresh)
        if det.class_id is None or det.confidence is None or len(det) == 0:
            return Detections.create_empty(frame_id)
        return Detections(
            bboxes=det.xyxy.astype(np.float32),
            scores=det.confidence.astype(np.float32),
            class_ids=det.class_id.astype(np.int64),
            frame_id=frame_id,
        )

In [22]:
from rfdetr import RFDETRNano

model = RFDETRNano()
model.optimize_for_inference()
rfdetr_detector = RFDETRAdapter(model, score_thresh=0.6)


Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In [25]:
name = "jets-1"

Path("results").mkdir(exist_ok=True)

out = track_video(rfdetr_detector, f"videos/{name}.mp4", f"results/{name}_tracked.mp4", class_names=rfdetr_detector.class_names)
display(Video(str(out), embed=True, width=720))

jets-1.mp4: 630frame [00:23, 27.31frame/s]


jets-1.mp4: 630 frames -> results/jets-1_tracked.mp4
